# Lab 2: Observability and Evaluation

A response can look correct while taking too long or making unnecessary model calls. **Observability** records what happened inside the application so you can locate that cost. **Evaluation** asks whether the result solved the task. You need both before calling a change an optimization.

In this lab, follow a TechMart returns question from request to answer, inspect its trace, calculate its model cost, and check the answer against a reference. Then connect the local record to the backend view and optionally extend the conversation.

## Learning Objectives

By the end of this notebook, you will be able to:

- Distinguish traces, spans, and sessions
- Read input/output/cache usage and estimate model cost
- Evaluate an observed answer against a reference
- Connect results to prompt versions for comparison

Trace a request, inspect its usage, and assess the response with **AgentCore evaluation**. You can also view the request in your Langfuse project. Application and evaluator calls have separate usage and charges.


## Before starting: use a fresh kernel

Complete [Jupyter setup](00-jupyter-notebook-101.ipynb) and [Prompts 101](01-prompts-101.ipynb). Select **Bedrock Workshop (Python 3.13)**, restart the kernel, and run from the top so tracing is initialized before the clients it observes.

The workshop has prepared both observability backends. On the AWS path, **traces** are stored in `aws/spans`; the notebook's **logs and metrics** use `/aws/bedrock-agentcore/workshop/notebook-agents`. These are different views of the same application: spans show timed work and its relationships, while logs record application messages and metrics summarize activity.


### Use the prepared environment

The notebook, AWS SDK, OpenTelemetry, and Langfuse dependencies are installed together in the Python 3.13 environment. Local kernel setup is described in [Jupyter Notebook 101](00-jupyter-notebook-101.ipynb).

Start with `OBSERVABILITY_BACKEND=agentcore`. Later in the notebook, you can select `langfuse` or `both` using the supplied Langfuse settings and a kernel restart. Switching backends needs no additional installation or infrastructure provisioning.


In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

if sys.version_info[:2] != (3, 13):
    raise RuntimeError("Select Bedrock Workshop (Python 3.13), then restart the kernel.")

REPO = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "workshop_utils").is_dir()),
    None,
)
if REPO is None:
    raise RuntimeError("Open this notebook inside the workshop repository.")

print("Kernel:", sys.executable)
print("Python:", sys.version.split()[0])
print({name: version(name) for name in ("boto3", "botocore", "bedrock-agentcore")})
import workshop_utils

print("Shared package:", workshop_utils.__file__)


## Select the experiment's model

The next cell chooses the workhorse and small-model roles before telemetry begins. On an uncached setup it makes small billed selection probes, preferring Sonnet 5 and Haiku 4.5, with Sonnet 4.6 as the same-provider workhorse fallback when unavailable. It then records the selected model and pricing settings in the experiment manifest.

Read that manifest before continuing. The model used to answer the customer must remain identifiable when you inspect its cost and quality later. Setup probes are recorded separately from the application request you will trace.


In [ ]:
import json
import os
import time
from uuid import uuid4

import boto3
from botocore.config import Config
from dotenv import load_dotenv
from IPython.display import display

from workshop_utils.models import Support
from workshop_utils.pacing import attach_boto3, get_workshop_pacer
from workshop_utils.preflight import preflight_models
from workshop_utils.pricing import get_price

load_dotenv(REPO / ".env", override=False)
REGION = "us-east-1"
# Explicit setup I/O: STS identity plus small billable probes on cache misses.
# Probe attempts are setup overhead and are not baseline usage or latency.
REFRESH_PREFLIGHT = False  # After permission changes, restart and set True for a new experiment.
PREFLIGHT_CACHE = REPO / ".models.json"
model_override = os.getenv("WORKSHOP_MODEL_ID") or None
probe_runtime = boto3.client(
    "bedrock-runtime", region_name=REGION,
    config=Config(connect_timeout=5, read_timeout=90, retries={"total_max_attempts": 1}),
)
attach_boto3(probe_runtime, get_workshop_pacer())
probe_sts = boto3.client(
    "sts", region_name=REGION,
    config=Config(connect_timeout=5, read_timeout=10, retries={"total_max_attempts": 1}),
)
try:
    PREFLIGHT = preflight_models(
        probe_runtime, sts_client=probe_sts, region=REGION,
        overrides={"workhorse": model_override} if model_override else None,
        cache_path=PREFLIGHT_CACHE, refresh=REFRESH_PREFLIGHT,
    )
finally:
    probe_runtime.close()
    probe_sts.close()

SELECTED = PREFLIGHT.models["workhorse"]
MODEL_ID = SELECTED.model_id
EFFORT = "low" if SELECTED.capabilities.support("effort") is Support.SUPPORTED else None
PREFLIGHT_REPORT = PREFLIGHT.as_dict()
display(PREFLIGHT_REPORT)
print("New probe attempts (separate setup overhead):",
      sum(not item["cache_hit"] for item in PREFLIGHT_REPORT["probes"]))
print("Probe cache:", PREFLIGHT_CACHE)

price = get_price(SELECTED)
MANIFEST = {
    **SELECTED.as_dict(), "effort": EFFORT, "service_tier": "standard",
    "prompt_version": "return-policy-v1", "dataset_version": "return-policy-smoke-v1",
    "pricing_date": price.verified_on, "price_evidence": price.evidence,
    "tools": [], "knowledge_base": None, "setup_preflight": PREFLIGHT_REPORT,
}
display(MANIFEST)


## 1. Initialize telemetry before the application

**Instrumentation** adds observations around application operations. **OpenTelemetry (OTEL)** supplies a common span format; **ADOT** is the AWS distribution used to export it. The helper below initializes telemetry and gives this notebook the service name `fundamentals-notebook`.

`agentcore` exports to AWS and keeps a local copy for inspection. `both` also attaches the Langfuse exporter; `langfuse` selects that destination alone. Initialize before making instrumented calls. Changing an initialized backend requires a kernel restart because the providers live for the Python process.

The lab records full prompt and response content, using the supplied synthetic policy and customer scenario. The local copy lets you inspect a finished call immediately; the backend adds a searchable view across requests.


In [ ]:
import os

from dotenv import load_dotenv

from workshop_utils.observability import resolve_notebook_backend, setup_observability

load_dotenv(REPO / ".env", override=False)
REGION = "us-east-1"
BACKEND = resolve_notebook_backend(REPO / ".env")
if BACKEND == "none":
    raise ValueError("This tracing lab needs agentcore, or an explicitly configured optional backend.")
telemetry = setup_observability(
    BACKEND, service_name="fundamentals-notebook", region=REGION,
    max_spans=2048,
)
LOG_GROUP = telemetry.log_group  # Custom logs/metrics group from the canonical environment/default.
TRACE_LOG_GROUP = "aws/spans"
print({"backend": telemetry.backend, "service": telemetry.service_name,
       "region": telemetry.region, "logs_metrics_log_group": LOG_GROUP,
       "trace_log_group": TRACE_LOG_GROUP})


### Trace, span, session, and process

A **trace typically represents one application request**, from its input to its output. Each timed step inside it is a **span**. Spans can nest: an agent request can contain a retrieval span and several model-call spans. A **session** groups related requests, such as the turns in one support conversation.

For example, “Can I return these headphones?” and “What if I lost the packaging?” belong to one customer session but create two request traces. If the first request queries a policy and then calls a model, those operations appear as children in its trace. Langfuse calls model-generation observations *generations*; they carry model, prompt, completion, and usage attributes.

| Term | What it identifies in this lab |
|---|---|
| Span | One timed operation, such as the wrapped Converse call |
| Trace | A request and its related spans, joined by `traceId` and parent IDs |
| Session | A conversation ID that can contain several request traces |
| Local collector | Finished spans emitted **inside this Python process** |
| Backend | A destination that ingests and stores exported spans |

A call to `invoke_agent_runtime` does not make the Runtime's model/tool spans appear in this notebook's collector. That remote process must export its own telemetry. A local wrapper can observe a remote response without observing the remote agent's reasoning/tool trajectory.

The wrapper uses a documented generic instrumentation scope and `gen_ai.task.input`/`gen_ai.task.output`, so its Q/A can be evaluated. A raw botocore span alone is not a substitute for this evaluable application span.


In [ ]:
import pandas as pd

from workshop_utils.bedrock import NormalizedUsage, build_converse_request, normalize_usage
from workshop_utils.evals import quick_eval
from workshop_utils.metrics import metrics_records
from workshop_utils.observability import response_span, traced_converse
from workshop_utils.pacing import attach_boto3, get_workshop_pacer
from workshop_utils.pricing import (
    AmbiguousCacheUsageError,
    InferredPriceError,
    UnknownPriceError,
    calculate_cost,
)

# A fresh application client is created after ADOT initialization.
# SELECTED was already resolved by the explicit preflight, not by offline ordering.
runtime = boto3.client(
    "bedrock-runtime", region_name=REGION,
    config=Config(connect_timeout=5, read_timeout=90, retries={"total_max_attempts": 1}),
)
attach_boto3(runtime, get_workshop_pacer())
display({**MANIFEST, "client_endpoint": runtime.meta.endpoint_url})


## 2. Run one bounded application request

The local fixture is a policy supplied directly to the model. It is **not a Knowledge Base retrieval or a tool invocation**. Those operations need their own real spans in the Developer Journey.

Predict the required answer before executing: the customer is within the window and has a receipt, but must also have original packaging. The criterion is written independently of the model's response.

The next cell makes **one billed inference request**, at low effort where supported. The helper records model ID, input/output/cache usage, and provider latency. It returns `_trace` correlation IDs in a copy of the response.


In [ ]:
POLICY = "TechMart accepts headphone returns within 30 days with a receipt and original packaging."
QUESTION = "I bought headphones 14 days ago and have a receipt. Can I return them?"
EXPECTED = "Yes, the purchase is within 30 days and you have a receipt; original packaging is also required."
SESSION_ID = f"fundamentals-{uuid4().hex}"
request = build_converse_request(
    SELECTED,
    [{"role": "user", "content": [{"text": QUESTION}]}],
    system=[{"text": (
        "You are a TechMart support assistant. Answer using only the policy in at most two sentences. "
        "State any requirement the customer has not confirmed. Policy: " + POLICY
    )}],
    max_tokens=512, effort=EFFORT,
)
# Clear local setup capture, then measure only this new application session/trace.
# Setup probes and their charges remain outside baseline measurements.
telemetry.collector.clear()
start_ns = time.time_ns()
start_clock = time.perf_counter()
response = traced_converse(runtime, session_id=SESSION_ID, **request)
client_e2e_ms = (time.perf_counter() - start_clock) * 1000
end_ns = time.time_ns()
answer = "".join(block["text"] for block in response["output"]["message"]["content"] if "text" in block)
TRACE_ID = response["_trace"]["trace_id"]
SPAN_ID = response["_trace"]["span_id"]
print(answer)
print("Stop reason:", response["stopReason"])
print("Correlation:", response["_trace"])
print("Client end-to-end ms:", round(client_e2e_ms, 1))


### Read the trace tree

The table below shows span names, parent links, scopes, and operations. Inspect `call_spans` for start/end times and the full attributes. The parent links tell you which operations belong to the same request; durations show where the wait occurred. Open the model-call span to find its prompt, response, and token counters.

One call can be observed by more than one instrumentation layer—for example, a workshop wrapper and its underlying botocore call. Those are two views of the same work. Adding both token fields would double the apparent bill, so the next step identifies actual model calls before aggregating usage.


In [ ]:
call_spans = [
    doc for doc in telemetry.collector.span_documents()
    if doc.get("traceId") == TRACE_ID
]
display(pd.DataFrame([
    {"span_id": doc["spanId"], "parent_id": doc.get("parentSpanId"),
     "scope": doc.get("scope", {}).get("name"), "name": doc.get("name"),
     "operation": doc.get("attributes", {}).get("gen_ai.operation.name")}
    for doc in call_spans if "startTimeUnixNano" in doc
]))


## 3. Turn spans into model-call costs

`metrics_records()` produces one row per attributed model call. It combines useful observations while excluding duplicate wrappers and aggregate agent totals already represented by child calls.

Input counters also need consistent meaning: Converse reports uncached input separately from cache reads/writes, while Strands 1.57's input counter includes cached input. The normalizer converts these into separate meters before pricing them with the shared catalog.

Read the output row as a small receipt: which model ran, how much input was processed, how much came from cache, how much output was generated, and what the known token rates imply. A non-streaming call has no measured TTFT. Missing rates or cache-write TTL evidence leave cost unknown.


In [ ]:
def price_observed_call(*, model_id, input_tokens, output_tokens, cache_read_tokens, cache_write_tokens):
    usage = NormalizedUsage(input_tokens, output_tokens, cache_read_tokens, cache_write_tokens)
    try:
        return calculate_cost(model_id, usage, region=REGION).total_cost
    except (UnknownPriceError, InferredPriceError, AmbiguousCacheUsageError):
        return None

records = metrics_records(call_spans, backend="memory", pricing=price_observed_call)
display(pd.DataFrame(records))
if len(records) != 1:
    raise RuntimeError("Expected one attributed model call. Inspect scope and parent IDs before trusting totals.")


### Reconcile against the actual response

The model response is the nearest available usage record for this call. Compare it with the normalized span record. Missing counts, disagreements, or unexpected extra calls are findings to investigate—not zeros to hide.

The cost estimate covers these observed token meters. It does not include evaluator calls, infrastructure, or unobserved failures/retries. To compute cost per solved task later, include all attempts and relevant tool/model work, then divide by tasks that satisfy the declared success criterion.


In [ ]:
raw_usage = normalize_usage(response["usage"], source="converse")
comparison = {
    name: {"response": getattr(raw_usage, name), "span": records[0][name]}
    for name in ("input_tokens", "output_tokens", "cache_read_tokens", "cache_write_tokens")
}
display(comparison)
assert all(values["response"] == values["span"] for values in comparison.values()), "Usage mismatch"
assert records[0]["ttft_ms"] is None, "Non-streaming Converse must not fabricate TTFT"
print("Token meters reconciled for this observed call.")


## 4. Immediate response check with ground truth

A **reference answer** describes what a correct response must contain, independently of what the model happened to say. For our return question, the answer must account for the 30-day window, receipt, and original packaging. Writing that reference first prevents a plausible-sounding answer from becoming its own grading standard.

An LLM evaluator compares the observed answer with that reference and returns a score plus explanation. Scores are useful for comparing prompt versions, but you should also read the underlying answer: a number alone cannot explain a missing policy condition.

`quick_eval()` evaluates **supplied span documents**. It does not invoke your application, fetch CloudWatch spans, or wait for log ingestion. It does call AgentCore's synchronous evaluator and consumes judge tokens.

Here we build a reduced generic Q/A document for the observed call using its correlation IDs and observation times. This checks whether the answer matches the reference. It evaluates response correctness; remote tool sequences and retrieval require their own spans. Passing this one example is a **smoke check**, not proof that quality is preserved across customers.

Source: [On-demand evaluation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-on-demand.html) and [generic framework spans](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/supported-frameworks-generic.html).


In [ ]:
qa_document = response_span(
    QUESTION, answer,
    session_id=SESSION_ID, trace_id=TRACE_ID, span_id=SPAN_ID,
    start_time_ns=start_ns, end_time_ns=end_ns,
    service_name="fundamentals-notebook",
)
display(qa_document)
# This creates a document only; it does not export an additional span.


### Evaluate the recorded answer

The previous cell prepared a Q/A span without calling an evaluator. The next cell uses `quick_eval` to send it to the native **`bedrock-agentcore.evaluate`** API: `evaluationInput.sessionSpans` carries the document, `evaluationTarget.traceIds` selects the trace, and `evaluationReferenceInputs` supplies `EXPECTED` for that session and trace.

The helper validates the request and returned result. Read the correctness score, explanation, and evaluator usage; this cell makes one billed evaluation call and does not rerun the support request.


In [ ]:
# One billed evaluator call when you execute this cell; no application inference.
evaluation_client = boto3.client(
    "bedrock-agentcore", region_name=REGION,
    config=Config(connect_timeout=5, read_timeout=90, retries={"total_max_attempts": 1}),
)
evaluation_started = time.perf_counter()
evaluation_results = quick_eval(
    [qa_document], client=evaluation_client, evaluator_id="Builtin.Correctness",
    session_id=SESSION_ID, trace_ids=[TRACE_ID], ground_truth={TRACE_ID: EXPECTED},
)
evaluation_elapsed_s = time.perf_counter() - evaluation_started
display(pd.DataFrame(evaluation_results))
print(f"Observed evaluator duration: {evaluation_elapsed_s:.2f} seconds")


Read the score **and its explanation**, then compare them with the answer and policy yourself. An evaluator can be wrong. The helper raises on service errors, missing target results, or ignored ground-truth fields so those cases cannot become a passing score.

Judge `tokenUsage` is separate from application `usage`; account for its charges separately from the model-call row. Keep evaluator identity, reference input, prompt/dataset version, exact model/profile, and observed duration with the result. More scenarios should include boundary dates, missing receipts, missing packaging, and requests outside policy; reserve some for held-out validation.

**Exercise:** predict how omitting the packaging requirement changes correctness. Make that change in a separately labelled prompt version, run the same scenario, and inspect both results. Do not edit the expected answer to fit the model's output.


In [ ]:
quality_record = {
    **MANIFEST, "session_id": SESSION_ID, "trace_id": TRACE_ID,
    "question": QUESTION, "expected_response": EXPECTED, "actual_response": answer,
    "stop_reason": response["stopReason"], "application_usage": response["usage"],
    "application_cost_usd": records[0]["cost_usd"], "client_e2e_ms": client_e2e_ms,
    "evaluator_results": evaluation_results, "evaluation_elapsed_s": evaluation_elapsed_s,
    "claim_scope": "one response-quality smoke scenario",
}
display(quality_record)


## 5. Find the request in CloudWatch

Open **CloudWatch → GenAI Observability** in `us-east-1`. Find service `fundamentals-notebook`, then locate this session and trace ID. Expand the spans and compare the request, response, usage, and duration with the local record.

This is the value of tracing beyond a notebook: many requests can be searched by identity, and a slow or expensive one can be opened to see which step caused it. Logs/metrics use the configured notebook group; the trace itself is in **`aws/spans`**.

The next cell flushes pending traces. Export and ingestion take time, so a local span can be visible before its CloudWatch copy. The optional readback allows up to five minutes for this trace to appear. In Langfuse-only mode, use the Langfuse view instead.


In [ ]:
flushed = telemetry.flush(timeout_ms=5000)
print({"trace_processors_flushed": flushed, "delivery_confirmed": False,
       "region": REGION, "trace_log_group": TRACE_LOG_GROUP, "logs_metrics_log_group": LOG_GROUP,
       "session_id": SESSION_ID, "trace_id": TRACE_ID})


### Read back this trace (optional)

The following cell queries the backend for the spans captured by this notebook. Matching trace and span IDs connects the backend copy to the request you just measured; it also makes the same documents available for later analysis.

The helper waits up to **300 seconds**, checking at five-second intervals for the complete expected set. A timeout means the full trace was not available within that interval. The optional exercise can therefore add five minutes to the lab.

This readback covers an application running in the notebook. A deployed Runtime agent emits its own spans from a different process, which is the next step in the Developer Journey.


In [ ]:
from datetime import UTC, datetime, timezone

from workshop_utils.evals import IncompleteTracesError, SpanCollectionError, collect_runtime_spans

RUN_CLOUDWATCH_READBACK = False
if RUN_CLOUDWATCH_READBACK and BACKEND in {"agentcore", "both"}:
    logs_client = boto3.client(
        "logs", region_name=REGION,
        config=Config(connect_timeout=2, read_timeout=5, retries={"total_max_attempts": 1}),
    )
    expected_ids = sorted({doc["spanId"] for doc in call_spans if "startTimeUnixNano" in doc})
    try:
        exported_spans = collect_runtime_spans(
            client=logs_client, session_id=SESSION_ID,
            expected_span_ids={TRACE_ID: expected_ids}, log_groups=list(dict.fromkeys([TRACE_LOG_GROUP, LOG_GROUP])),
            start_time=datetime.fromtimestamp(start_ns / 1e9, UTC),
            end_time=datetime.fromtimestamp(end_ns / 1e9, UTC),
            timeout_s=300, poll_interval_s=5,
        )
    except (IncompleteTracesError, SpanCollectionError) as exc:
        print("CloudWatch readback not complete:", exc)
    else:
        cloudwatch_records = metrics_records(exported_spans, backend="cloudwatch", pricing=price_observed_call)
        display(pd.DataFrame(cloudwatch_records))
        local_by_id = {(row["trace_id"], row["span_id"]): row for row in records}
        remote_by_id = {(row["trace_id"], row["span_id"]): row for row in cloudwatch_records}
        assert local_by_id.keys() == remote_by_id.keys(), "Model-call identity mismatch"
        for key in local_by_id:
            for meter in ("input_tokens", "output_tokens", "cache_read_tokens", "cache_write_tokens"):
                assert local_by_id[key][meter] == remote_by_id[key][meter], f"Mismatch: {key}, {meter}"
        print("Exported model-call identities and token meters reconciled for this trace.")
else:
    print("CloudWatch readback skipped; no delivery claim.")


### How this extends to a deployed agent

Calling a remote Runtime returns its answer, but does not copy the remote tool/model spans into the notebook's collector. To investigate that agent, collect its exported spans by session ID and invocation window from **`aws/spans`**, alongside its application logs when needed.

This distinction determines what you can evaluate. The returned answer supports an immediate response-correctness check. Assessing whether the agent retrieved the right evidence or called the right tools requires its actual remote spans and their conversation/tool content. Keep the time spent collecting traces separate from the agent's response time.

The Developer Journey applies these ideas to deployment and real tool execution. Here, the policy was supplied directly in the prompt.


## 6. Optional two-turn exercise

Use one session ID for the conversation and a separate trace for each request. Replay the **complete** assistant message, including any preserved signed reasoning blocks, then append the next user message. Do not reconstruct history from displayed text or edit prior blocks on models requiring append-only preserved thinking.

This optional cell makes one additional application call; it does not run another evaluator. Inspect how the second trace relates to the same session, then design a reference answer for the new question before evaluating it.


In [ ]:
RUN_TWO_TURN_EXERCISE = False
if RUN_TWO_TURN_EXERCISE:
    messages = [
        *request["messages"],
        response["output"]["message"],
        {"role": "user", "content": [{"text": "What if I no longer have the original packaging?"}]},
    ]
    turn_two_request = build_converse_request(
        SELECTED, messages, system=request["system"], max_tokens=512, effort=EFFORT,
    )
    turn_two = traced_converse(runtime, session_id=SESSION_ID, **turn_two_request)
    print("".join(block["text"] for block in turn_two["output"]["message"]["content"] if "text" in block))
    print(turn_two["_trace"])
    turn_two_spans = [doc for doc in telemetry.collector.span_documents()
                      if doc.get("traceId") == turn_two["_trace"]["trace_id"]]
    display(pd.DataFrame(metrics_records(turn_two_spans, backend="memory", pricing=price_observed_call)))
    telemetry.flush(timeout_ms=5000)
else:
    print("Two-turn exercise skipped; no extra application or evaluator call.")


## 7. Explore traces in Langfuse

**Langfuse is an open-source LLM observability platform.** It groups model and tool observations into traces, lets you follow sessions across requests, and displays inputs, outputs, usage, and scores together.

To try the prepared service, select `OBSERVABILITY_BACKEND=both` for dual export or `langfuse` for Langfuse-only, then **restart the kernel and run from the top**. Follow **Langfuse Project Setup** in your workshop guide to create your project keys and save `LANGFUSE_BASE_URL`, `LANGFUSE_PUBLIC_KEY`, and `LANGFUSE_SECRET_KEY` in `.env`; no additional install is needed.

Open **Tracing**, find this trace ID, and expand its observations. In `both` mode, compare the same request in CloudWatch and Langfuse. The views should describe the same work, even if their displayed price estimates use different price tables. The AgentCore evaluator in this notebook remains the quality check for either trace destination.


### Prompt management: keep changes connected to results

**Prompt management is a systematic approach to storing, versioning, and retrieving prompts in LLM applications.** A version tells you exactly which instructions produced an answer. This lets you compare changes, reproduce a result, and return to earlier instructions when a candidate regresses.

For example, version 1 might say “answer the returns question,” while version 2 explicitly requires mentioning unconfirmed policy conditions. If version 2 scores better, you need its exact text and model settings—not just the label “optimized”—to use that improvement again.

This notebook records `prompt_version` and `dataset_version` in its manifest; it does not create a hosted prompt. The lifecycle labs extend that idea with versioned configuration and evaluation before promotion. Langfuse prompt management is another way to store versions and connect generations to them.

Sources: [Langfuse prompt management](https://langfuse.com/docs/prompt-management/overview), [Langfuse OpenTelemetry integration](https://langfuse.com/integrations/native/opentelemetry), [AgentCore observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html).


## 8. Finish the telemetry session

After the readback and optional exercises, run the next cell to close the notebook's telemetry providers. **Flush** exports pending traces; **shutdown** also stops the background log and metric exporters.

Save any results you need before shutdown clears the in-memory collector. When you later clean up workshop resources, stop exporting processes first so they do not continue writing. This cell stops telemetry only; restart the kernel to begin another tracing session.


In [ ]:
telemetry.shutdown()
print("Telemetry exporters stopped. Restart the kernel before initializing telemetry again.")
# Resource deletion belongs to the separate teardown workflow, after exporters have stopped.

## Bring the measurements into the Playbook

You followed one request through local capture, model-call accounting, a reference-based quality check, and backend tracing. Use the trace tree to explain where the cost and latency came from; use the answer and evaluator explanation to decide whether the task was solved.

For a new prompt version, repeat the same scenario and retain its model/settings, usage, timing, and quality result. Then test additional held-out scenarios before generalizing from one success.

**Checkpoint:** explain why a session can contain several traces, why one model call can produce several spans, and why evaluating tool choice needs more than the final answer.

Continue to the [Optimization Playbook](../02-optimization-playbook/01-low-effort.ipynb) to change one lever at a time, or the [Developer Journey](../03-developer-journey/01-baseline-agent.ipynb) to follow a complete support agent.


## References

- [AgentCore observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html)
- [On-demand evaluation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-on-demand.html)
- [Generic framework spans](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/supported-frameworks-generic.html)
- [AgentCore Evaluate API](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore/client/evaluate.html)
- [Langfuse OpenTelemetry integration](https://langfuse.com/integrations/native/opentelemetry)
- [Langfuse prompt management](https://langfuse.com/docs/prompt-management/overview)
